In [ ]:
import numpy as np
import bacco
import matplotlib.pyplot as plt

import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
## Load the Zooms
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']
#name_list = ['bf_sim']

snap = 264
zoom = {}

loaded = []
for i in range(len(name_list)):
    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)

### Load the Halo Selection ###
xmatch = {}
for i in range(len(name_list)):
    xmatch[name_list[i]] = utils.cross_match(zoom[name_list[i]], snap=264, name=name_list[i])


In [ ]:
def get_contamination_fraction(zoom, xmatch):

    # The high-resolution mass will be sum of the DM, gas, stars and BH mass
    hd_mass = (zoom.fof['halo_mfof_type'][:,0] + zoom.fof['halo_mfof_type'][:,1] + zoom.fof['halo_mfof_type'][:,4] + zoom.fof['halo_mfof_type'][:,5])[xmatch['ind']]

    # The low resolution mass will be the sum of the low-res DM particles
    lr_mass = zoom.fof['halo_mfof_type'][:,2][xmatch['ind']] + zoom.fof['halo_mfof_type'][:,3][xmatch['ind']]

    return lr_mass / (hd_mass + lr_mass)    


In [ ]:
f_cont = {}
m_halo = {}
for i in range(len(name_list)):
    f_cont[name_list[i]] = get_contamination_fraction(zoom[name_list[i]], xmatch[name_list[i]])
    m_halo[name_list[i]] = 1e10 * zoom[name_list[i]].fof['halo_mfof'][xmatch[name_list[i]]['ind']]

In [ ]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/dm_halo_sel_1pmbin.txt") as f:
    halo_sel = []

    for line in f.readlines():
        halo_sel.append(int(line.split()[0]))

halo_sel = np.array(halo_sel)

# Load MTNG and get the fraction of halos to do the upweighting
mtng_dm = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG-DM-Gadget4/MTNG-L500-2160-A/", snap=265)

In [ ]:
m_halo = 1e10 * mtng_dm.fof['halo_m200c'][halo_sel]

In [ ]:
print(m_halo[452-1] / 1e14)
print(m_halo[452-9] / 1e13)
print(m_halo[452-40] / 1e12)
print(m_halo[452-180] / 1e11)

In [ ]:
halo_sel = np.loadtxt("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/dm_halo_sel_1pmbin.txt", dtype=int)

In [ ]:
print(halo_sel[452-1])
print(halo_sel[452-9])
print(halo_sel[452-40])
print(halo_sel[452-180])

In [ ]:
print(f_cont['fiducial'][-1])
print(f_cont['fiducial'][-9])
print(f_cont['fiducial'][-40])
print(f_cont['fiducial'][-180])

In [ ]:
len(m_halo)